<a href="https://colab.research.google.com/github/kklmts/lab8-9/blob/main/8-9.1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import requests

# 1. Завантаження CSV з практичної роботи 5
URL = "https://raw.githubusercontent.com/kklmts/lab5-6/master/population.csv"

try:
    df = pd.read_csv(URL)
    print("Файл успішно завантажено")
except Exception as e:
    print("Помилка завантаження файлу:", e)
    raise SystemExit

# Очікувані стовпці: Country, Code, Year, Population
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
df["Population"] = pd.to_numeric(df["Population"], errors="coerce")

# Доповнення даних показниками Світового банку
def get_indicator(indicator, new_name, year_from, year_to):
    """Завантажує показник Світового банку через API
    і повертає DataFrame зі стовпцями Code, Year, <new_name>."""
    url = f"https://api.worldbank.org/v2/country/all/indicator/{indicator}"
    params = {
        "format": "json",
        "per_page": 20000,
        "date": f"{year_from}:{year_to}",
    }
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    records = response.json()[1]

    data = pd.DataFrame(
        {
            "Code": [r["countryiso3code"] for r in records],
            "Year": [int(r["date"]) for r in records],
            new_name: [r["value"] for r in records],
        }
    )
    data["Year"] = data["Year"].astype("Int64")
    return data


def get_country_info():
    """Довідник Світового банку: код країни, регіон, рівень доходу.
    Для груп країн (World, Low income, ...) регіон = 'Aggregates'."""
    url = "https://api.worldbank.org/v2/country"
    params = {"format": "json", "per_page": 500}
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    records = response.json()[1]

    return pd.DataFrame(
        {
            "Code": [r["id"] for r in records],
            "Region": [r["region"]["value"].strip() for r in records],
            "Income_level": [r["incomeLevel"]["value"].strip() for r in records],
        }
    )


y_min, y_max = int(df["Year"].min()), int(df["Year"].max())

try:
    gdp = get_indicator("NY.GDP.PCAP.CD", "GDP_per_capita", y_min, y_max)
    life = get_indicator("SP.DYN.LE00.IN", "Life_expectancy", y_min, y_max)
    info = get_country_info()
except Exception as e:
    print("Помилка завантаження даних Світового банку:", e)
    raise SystemExit

# Додаємо нові стовпці до основної таблиці
df = df.merge(gdp, on=["Code", "Year"], how="left")
df = df.merge(life, on=["Code", "Year"], how="left")
df = df.merge(info, on="Code", how="left")

# У файлі є не лише країни, а й групи країн (World, High income тощо). Для коректного аналізу залишаємо тільки країни.
df = df[df["Region"] != "Aggregates"].reset_index(drop=True)

# Зберігаємо доповнений набір даних
df.to_csv("population_extended.csv", index=False)
print("Доповнений набір даних збережено: population_extended.csv")

# Базовий аналіз
print("\nПочаткові дані:")
print(df)

print("\nПерші 3 рядки DataFrame:")
print(df.head(3))

print("\nТипи даних:")
print(df.dtypes)

print("\nРозмір DataFrame (рядки, стовпці):")
print(df.shape)

print("\nОписова статистика:")
print(df.describe())

# Попереднє оброблення. Зміна типів даних
df["Country"] = df["Country"].astype("string")
df["Code"] = df["Code"].astype("category")
df["Year"] = df["Year"].astype(int)
df["Region"] = df["Region"].astype("category")
df["Income_level"] = df["Income_level"].astype("category")

print("\nТипи даних після перетворення:")
print(df.dtypes)

# Нові розрахункові стовпці
df["Population_mln"] = df["Population"] / 1e6                      # населення, млн
df["GDP_total_bln"] = df["Population"] * df["GDP_per_capita"] / 1e9  # ВВП, млрд USD

print("\nDataFrame з новими показниками:")
print(df)

# Фільтрація: країни з населенням понад 50 млн осіб
filtered_df = df[df["Population"] > 50_000_000]
print("\nРезультат фільтрації (населення > 50 млн):")
print(filtered_df)

# Сортування: за населенням від більшого до меншого
sorted_df = df.sort_values(by="Population", ascending=False)
print("\nРезультат сортування за населенням:")
print(sorted_df)

# Групування та агрегація (за регіоном)
grouped_df = (
    df.groupby("Region", observed=True)
    .agg(
        Countries=("Country", "count"),
        Avg_population_mln=("Population_mln", "mean"),
        Max_population_mln=("Population_mln", "max"),
        Min_population_mln=("Population_mln", "min"),
        Avg_GDP_per_capita=("GDP_per_capita", "mean"),
        Avg_life_expectancy=("Life_expectancy", "mean"),
    )
    .round(2)
    .reset_index()
    .sort_values(by="Avg_population_mln", ascending=False)
)

print("\nРезультат групування за регіоном:")
print(grouped_df)

# Кількість унікальних значень
print("\nКількість унікальних країн:", df["Country"].nunique())
print("Кількість унікальних регіонів:", df["Region"].nunique())
print("Кількість унікальних рівнів доходу:", df["Income_level"].nunique())
print("Кількість унікальних років:", df["Year"].nunique())

# Збереження результатів у новий CSV
grouped_df.to_csv("analysis_results.csv", index=False)
print("\nРезультати збережено у файл analysis_results.csv")

Файл успішно завантажено
Доповнений набір даних збережено: population_extended.csv

Початкові дані:
                   Country Code  Year  Population  GDP_per_capita  \
0              High income  NaN  2019  1389767299             NaN   
1               Low income  NaN  2019   652255298             NaN   
2      Lower middle income  NaN  2019  2724611683             NaN   
3      Upper middle income  NaN  2019  3010528196             NaN   
4              Afghanistan  AFG  2019    37856121      496.602504   
..                     ...  ...   ...         ...             ...   
216  Virgin Islands (U.S.)  VIR  2019      106669    38633.529892   
217     West Bank and Gaza  PSE  2019     4685306     3656.858271   
218            Yemen, Rep.  YEM  2019    35111408             NaN   
219                 Zambia  ZMB  2019    18513839     1258.986198   
220               Zimbabwe  ZWE  2019    15271368     2184.521554   

     Life_expectancy                                             Region